# Final Project: Multi-LLM Collaborative Debate System

**Course:** Applied LLM Systems

The idea of this system is to fight hallucinations and reasoning errors through debate between multiple LLMs.
Instead of trusting a single model, several models solve the same problem, criticize each other,
fix their solutions based on the feedback, and finally an independent judge picks the best answer.

The pipeline has these stages:

- **Stage 0** - Role self-assessment: every agent looks at the problem and says how confident it would be as a Solver vs as a Judge
- **Stage 0.5** - Deterministic role assignment: a simple algorithm turns those self-assessments into 3 Solvers + 1 Judge
- **Stage 1** - The 3 Solvers solve the problem independently (no communication between them)
- **Stage 2** - Peer review: each Solver reviews the other two solutions with structured feedback
- **Stage 3** - Refinement: each Solver answers every critique (accept and fix, or reject and defend) and produces a refined solution
- **Stage 4** - The Judge sees everything (originals, reviews, refined solutions), verifies the reasoning and picks a winner

The winner's refined answer is the system's final answer.

## The 4 agents - 2 providers, strong + weak model from each

The agents are 4 different models: from each provider I take one strong modern model and one
deliberately weaker/older model:

| Agent | Provider | Model | Tier |
|---|---|---|---|
| openai | OpenAI | gpt-4.1-mini | mid (2025) |
| openai-35 | OpenAI | gpt-3.5-turbo | weak (2023) |
| claude | Anthropic | claude-sonnet-4-5 | strong |
| claude-haiku | Anthropic | claude-haiku-4-5 | small/fast |

The weak models are a DELIBERATE choice, not a budget one. In an early version all four agents
were frontier-tier models and every metric saturated at ~100% accuracy - you could not see the
debate doing anything. With a strong/weak mix the interesting dynamics become measurable: weak
solvers produce real errors, peer review has something to catch, refinement has something to
fix, and the judge has real disagreements to resolve.

(Historical note: the original lineup was OpenAI + Anthropic + Google Gemini + Llama-on-Groq,
i.e. 4 separate providers. The Gemini and Groq free-tier keys kept dying mid-run - exhausted
daily quotas and rate limits - so both slots were replaced with models from the two reliable
paid providers. The assignment explicitly allows using one provider with different models.)

All agents are called through the same OpenAI Python SDK: Anthropic exposes an OpenAI-compatible
endpoint, so the only per-agent configuration is `base_url` + key + model name. gpt-3.5-turbo
predates structured outputs (no json_schema support), so it runs through the fallback path in
`ask()`: plain JSON mode + manual Pydantic validation.

Which agent becomes the Judge is decided per problem in Stage 0.5.

**Note:** running this notebook end to end makes a lot of API calls (around 25-30 per problem, 25 problems).
Results are checkpointed to `results/debate_results.json` after every problem, so if it crashes you can just
re-run the cell and it continues where it stopped.

In [33]:
import json
import os
import time
from typing import List, Literal
from concurrent.futures import ThreadPoolExecutor

from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads all 4 API keys from the .env file (never hardcode keys!)

# one config entry per agent: which endpoint, which key, which model,
# and whether the endpoint supports native structured outputs (json_schema)
AGENTS = {
    "openai": {
        "model": "gpt-4.1-mini",
        "base_url": None,  # default OpenAI endpoint
        "api_key": os.environ["OPENAI_API_KEY"],
        "structured": True,
    },
    "claude": {
        "model": "claude-sonnet-4-5",
        "base_url": "https://api.anthropic.com/v1/",
        "api_key": os.environ["ANTHROPIC_API_KEY"],
        "structured": True,
    },
    "openai-35": {
        # deliberately an OLD model (2023 era): a weak solver makes the debate mechanism visible.
        # it also has no json_schema support, so it exercises the fallback path in ask()
        "model": "gpt-3.5-turbo",
        "base_url": None,
        "api_key": os.environ["OPENAI_API_KEY"],
        "structured": False,
    },
    "claude-haiku": {
        "model": "claude-haiku-4-5",
        "base_url": "https://api.anthropic.com/v1/",
        "api_key": os.environ["ANTHROPIC_API_KEY"],
        "structured": True,
    },
}

# one OpenAI SDK client per provider.
# timeout + max_retries=0 matter: the SDK default is a 10 MINUTE timeout with internal retries,
# so one unresponsive provider can silently stall the whole run for an hour. Our own retry loop
# in ask() handles retries with proper backoff instead.
clients = {
    name: OpenAI(api_key=cfg["api_key"], base_url=cfg["base_url"], timeout=120, max_retries=0)
    for name, cfg in AGENTS.items()
}

# separate model used only for grading answers against ground truth (it is not part of the debate)
GRADER_MODEL = "gpt-4.1-mini"
grader_client = clients["openai"]

RESULTS_PATH = "results/debate_results.json"
BASELINE_PATH = "results/baseline_results.json"
os.makedirs("results", exist_ok=True)

AGENT_NAMES = list(AGENTS.keys())
print("agents:", {n: AGENTS[n]["model"] for n in AGENT_NAMES})

agents: {'openai': 'gpt-4.1-mini', 'claude': 'claude-sonnet-4-5', 'openai-35': 'gpt-3.5-turbo', 'claude-haiku': 'claude-haiku-4-5'}


## Phase 1: Problem dataset

The dataset lives in `problems.json`: 25 problems in 5 categories. I deliberately did NOT use
the categories suggested in the assignment (those are what most projects will use). The design
goals for the problems:

1. **multi-step derivations, not lookups** - every problem requires an actual computation with
   the given numbers (a payment formula, an energy balance, a characteristic equation), so a
   model cannot answer from memorized trivia; it has to carry the arithmetic through correctly
2. **multi-part answers** - most problems ask for 2-3 values (e.g. radius AND height AND area),
   which makes partially-right-but-wrong answers common and gives the peer reviewers something
   concrete to catch
3. **verifiable ground truth** - every answer was computed and verified with Python before
   being added to the dataset (the verification script computes each value independently)

The 5 categories, exam-style:

- **optimization** - calculus optimization and related rates (minimal can surface, fence along
  a river, open-top box, related-rates ladder)
- **probability_expectation** - expected values that need state equations or clever
  decompositions (HH vs HT waiting times, gambler's ruin, coupon collector, first ace position)
- **applied_physics** - multi-step numeric derivations (incline with friction, projectile from
  a cliff, spring launch + energy conservation, RC circuit charging)
- **finance** - annuity and discounting formulas with exact numbers (mortgage payment, NPV,
  doubling time, future value of monthly savings, effective annual rates)
- **discrete_math** - CRT system, Fermat's little theorem, Euler's totient, solving a linear
  recurrence via the characteristic equation, base-7 arithmetic with carries

In [34]:
with open("problems.json", "r", encoding="utf-8") as f:
    problems = json.load(f)

print(f"loaded {len(problems)} problems")

from collections import Counter
print(Counter(p["category"] for p in problems))

print("\nexample problem:")
print(problems[0]["question"])
print("ground truth:", problems[0]["answer"])

loaded 25 problems
Counter({'optimization': 5, 'probability_expectation': 5, 'applied_physics': 5, 'finance': 5, 'discrete_math': 5})

example problem:
A closed cylindrical can (top and bottom included) must hold exactly 500 cm^3. Using calculus, find the radius and height that minimize the total surface area, and compute that minimal surface area. Give radius, height and area numerically (2 decimal places), and state the relationship between h and r at the optimum.
ground truth: r approx 4.30 cm, h approx 8.60 cm, minimal surface area approx 348.73 cm^2; at the optimum h = 2r (height equals diameter)


## Structured output schemas

Every stage returns structured JSON. Like we did in the lectures, I define Pydantic models and pass them
as `response_format` to `client.chat.completions.parse()` - the API then guarantees valid JSON that
matches the schema. All four current models support this natively; for models that do not
(like the Llama agent this project originally had), the same Pydantic models are used to
validate manually parsed JSON, so the rest of the pipeline never notices the difference.

In [ ]:
class RoleAssessment(BaseModel):
    """Stage 0 output - agent self-assesses which role fits it for this problem"""
    preferred_role: Literal["solver", "judge"]
    solver_confidence: float  # 0..1
    judge_confidence: float   # 0..1
    reasoning: str


class Solution(BaseModel):
    """Stage 1 output - independent solution"""
    reasoning: str
    final_answer: str
    confidence: float


class ReviewError(BaseModel):
    """a single error found by a reviewer"""
    location: str
    error_type: Literal["calculation_error", "logical_error", "missing_case",
                        "wrong_formula", "misread_problem", "other"]
    description: str
    severity: Literal["minor", "major", "critical"]


class PeerReview(BaseModel):
    """Stage 2 output - structured review of one peer solution"""
    strengths: List[str]
    weaknesses: List[str]
    errors: List[ReviewError]
    suggested_changes: List[str]
    overall_assessment: Literal["correct", "promising_but_flawed", "fundamentally_wrong"]


class CritiqueResponse(BaseModel):
    """how the solver responded to one critique"""
    critique: str
    response: str
    accepted: bool


class Refinement(BaseModel):
    """Stage 3 output - refined solution after peer feedback"""
    changes_made: List[CritiqueResponse]
    refined_reasoning: str
    refined_answer: str
    confidence: float


class JudgeVerdict(BaseModel):
    """Stage 4 output"""
    winner: Literal["solver_1", "solver_2", "solver_3"]
    confidence: float
    reasoning: str


class GradeResult(BaseModel):
    """used by the grader model to compare an answer with ground truth"""
    is_correct: bool
    explanation: str


class MajorityResult(BaseModel):
    """used to cluster equivalent answers for the voting baseline and consensus metric"""
    all_same: bool
    majority_answer: str
    majority_size: int

## API helper

One wrapper for all calls, hiding the provider differences:

- **native path** (OpenAI models, Claude): `chat.completions.parse` with the Pydantic schema
- **fallback path** (for models without json_schema support, e.g. the Llama agent this
  project originally had): schema is injected into the system prompt, the model is forced into
  JSON mode with `response_format={"type": "json_object"}`, and the output is parsed and validated
  with Pydantic by hand - exactly what we had to do in the lectures before structured outputs existed

Retries use exponential backoff up to 60s because free-tier providers rate-limit
per minute, and a fixed short wait would just burn all the retries inside the same rate window.
`run_parallel` fires independent calls of one stage at the same time, otherwise a full run
takes forever.

In [ ]:
def ask(agent: str, system_prompt: str, user_prompt: str, schema, temperature: float = 0.3, retries: int = 6):
    """one structured-output call to any of the 4 providers, with retries + backoff"""
    cfg = AGENTS[agent]
    client = clients[agent]
    last_err = None
    feedback = ""  # validation feedback for the fallback path
    for attempt in range(retries):
        try:
            if cfg["structured"]:
                resp = client.chat.completions.parse(
                    model=cfg["model"],
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    response_format=schema,
                    temperature=temperature,
                )
                return resp.choices[0].message.parsed
            else:
                # fallback: JSON mode + manual pydantic validation.
                # important detail learned the hard way: if you just paste the JSON schema, llama
                # sometimes returns the SCHEMA itself instead of an instance of it. So the prompt
                # explicitly demands actual values, lists the expected keys, and on a validation
                # error the next attempt includes what exactly was wrong.
                schema_json = json.dumps(schema.model_json_schema())
                sys = (
                    system_prompt +
                    "\n\nRespond ONLY with a single JSON object containing your ACTUAL values "
                    "(an instance, NOT the schema itself). Top-level keys must be exactly: "
                    f"{list(schema.model_fields.keys())}. "
                    f"The object must validate against this JSON schema:\n{schema_json}" + feedback
                )
                resp = client.chat.completions.create(
                    model=cfg["model"],
                    messages=[
                        {"role": "system", "content": sys},
                        {"role": "user", "content": user_prompt},
                    ],
                    response_format={"type": "json_object"},
                    temperature=temperature,
                )
                raw = resp.choices[0].message.content
                data = json.loads(raw)
                # gpt-3.5 sometimes puts a dict or a number where the schema wants a string
                # (e.g. final_answer: {"payment": 843.86}) - stringify instead of failing
                if isinstance(data, dict):
                    for key, val in data.items():
                        field = schema.model_fields.get(key)
                        if field is not None and field.annotation is str and val is not None and not isinstance(val, str):
                            data[key] = json.dumps(val) if isinstance(val, (dict, list)) else str(val)
                return schema.model_validate(data)
        except Exception as e:
            last_err = e
            if type(e).__name__ in ("ValidationError", "JSONDecodeError"):
                feedback = (f"\n\nYour previous response was invalid: {str(e)[:300]}. "
                            f"Return only a valid instance with actual values.")
                wait = 2  # not a rate limit problem, no need for long backoff
            else:
                wait = min(60, 3 * 2 ** attempt)  # 3, 6, 12, 24, 48, 60 - survives per-minute rate limits
            print(f"  [retry] {agent} failed ({type(e).__name__}), waiting {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"all {retries} retries failed for {agent}: {last_err}")


def ask_grader(system_prompt, user_prompt, schema, retries: int = 4):
    """grading calls always go to the grader model with temperature 0 (with retries too -
    a single transient timeout here should not kill a whole problem)"""
    last_err = None
    for attempt in range(retries):
        try:
            resp = grader_client.chat.completions.parse(
                model=GRADER_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                response_format=schema,
                temperature=0.0,
            )
            return resp.choices[0].message.parsed
        except Exception as e:
            last_err = e
            wait = 3 * (attempt + 1)
            print(f"  [retry] grader failed ({type(e).__name__}), waiting {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"grader failed after {retries} retries: {last_err}")


def run_parallel(tasks):
    """tasks = list of zero-argument functions, runs them at the same time and returns results in order"""
    with ThreadPoolExecutor(max_workers=6) as ex:
        futures = [ex.submit(t) for t in tasks]
        return [f.result() for f in futures]


def clamp01(x: float) -> float:
    """some models occasionally return confidence like 99 instead of 0.99"""
    if x > 1:
        x = x / 100
    return max(0.0, min(1.0, x))

## Stage 0 + 0.5: role self-assessment and deterministic assignment

Every agent gets the problem and honestly rates itself for the Solver and Judge roles.
Then the assignment algorithm is fully deterministic:

1. the Judge is the agent with the highest `judge_confidence` (ties broken by a fixed agent order)
2. the remaining 3 agents become Solvers, ordered by their `solver_confidence` (solver_1 = most confident)

So given the same self-assessments the role distribution is always the same - no randomness.

In [37]:
ROLE_SYS = (
    "You are one of four LLM agents in a collaborative debate system. "
    "Three agents will act as Solvers (they solve the problem independently and review each other) "
    "and one agent will be the Judge (it evaluates all refined solutions and picks the best one). "
    "Honestly self-assess how well each role suits you for THIS specific problem. "
    "Give confidence scores between 0 and 1 for each role."
)


def assess_roles(problem: dict) -> dict:
    """Stage 0: ask all 4 agents which role they want for this problem"""
    prompt = f"Problem:\n{problem['question']}\n\nWhich role do you prefer for this problem and how confident are you in each role?"
    results = run_parallel([
        (lambda a=a: ask(a, ROLE_SYS, prompt, RoleAssessment, temperature=0.0)) for a in AGENT_NAMES
    ])
    return dict(zip(AGENT_NAMES, results))


def assign_roles(assessments: dict):
    """Stage 0.5: deterministic algorithm -> (judge_agent, [solver_agents ordered])"""
    judge = max(AGENT_NAMES, key=lambda a: (clamp01(assessments[a].judge_confidence), -AGENT_NAMES.index(a)))
    solvers = [a for a in AGENT_NAMES if a != judge]
    solvers.sort(key=lambda a: (-clamp01(assessments[a].solver_confidence), AGENT_NAMES.index(a)))
    return judge, solvers

## Stage 1: independent solutions

The three Solvers get only the problem text. They do not see each other at this point.

In [38]:
SOLVE_SYS = (
    "You are an expert problem solver. Solve the problem step by step, showing all your reasoning. "
    "Be careful with calculations and double-check them. Watch out for classic traps and counterintuitive "
    "cases. End with a clear and short final_answer that directly answers the question."
)


def solve_independently(problem: dict, solvers: list) -> dict:
    """Stage 1: returns {'solver_1': Solution, 'solver_2': ..., 'solver_3': ...}"""
    results = run_parallel([
        (lambda a=a: ask(a, SOLVE_SYS, problem["question"], Solution, temperature=0.3)) for a in solvers
    ])
    return {f"solver_{i+1}": r for i, r in enumerate(results)}

## Stage 2: peer review

Each Solver reviews the other two solutions -> 6 reviews total per problem.
The review is structured: strengths, weaknesses, concrete errors with location and severity,
and suggested changes. I explicitly tell the reviewer not to invent errors, because in early tests
reviewers sometimes "found" problems in perfectly correct solutions just to have something to say.

In [39]:
REVIEW_SYS = (
    "You are a rigorous peer reviewer in a debate system. Analyze the given solution carefully step by step. "
    "Look for calculation errors, logical leaps, missed edge cases and wrong formulas. Be specific about "
    "WHERE each error is. Be critical but fair - do NOT invent errors that are not there. If the solution "
    "is fully correct, say so."
)


def peer_review(problem: dict, solutions: dict, solver_agents: dict) -> dict:
    """Stage 2: each solver reviews the other two. returns {target_id: [(reviewer_id, PeerReview), ...]}"""
    ids = list(solutions.keys())
    pairs = [(r, t) for r in ids for t in ids if r != t]

    def make_task(reviewer_id, target_id):
        sol = solutions[target_id]
        prompt = (
            f"Problem:\n{problem['question']}\n\n"
            f"Solution written by {target_id}:\n"
            f"Reasoning: {sol.reasoning}\n"
            f"Final answer: {sol.final_answer}\n\n"
            f"Write your structured review of this solution."
        )
        return lambda: ask(solver_agents[reviewer_id], REVIEW_SYS, prompt, PeerReview, temperature=0.2)

    results = run_parallel([make_task(r, t) for r, t in pairs])

    reviews = {sid: [] for sid in ids}
    for (reviewer_id, target_id), review in zip(pairs, results):
        reviews[target_id].append((reviewer_id, review))
    return reviews

## Stage 3: refinement

Each Solver gets the two reviews of its own solution and must respond to every critique explicitly:
either accept it and fix the solution, or reject it and defend the original reasoning.
This "defend if the critique is wrong" part matters a lot - without it solvers just fold under
any criticism, even wrong criticism, and correct answers get destroyed.

In [40]:
REFINE_SYS = (
    "You are refining your own solution after receiving peer reviews. Address EVERY critique explicitly: "
    "either accept it and fix your solution accordingly, or reject it and explain exactly why the critique "
    "is wrong. Do not blindly agree with critiques - if your original reasoning was correct, defend it. "
    "Then output your final refined solution and refined answer."
)


def format_reviews(review_list) -> str:
    text = ""
    for reviewer_id, rv in review_list:
        text += f"\nReview from {reviewer_id} (assessment: {rv.overall_assessment}):\n"
        if rv.strengths:
            text += f"  strengths: {'; '.join(rv.strengths)}\n"
        if rv.weaknesses:
            text += f"  weaknesses: {'; '.join(rv.weaknesses)}\n"
        for e in rv.errors:
            text += f"  error at '{e.location}' [{e.severity} / {e.error_type}]: {e.description}\n"
        if rv.suggested_changes:
            text += f"  suggested changes: {'; '.join(rv.suggested_changes)}\n"
    return text if text else "\n(no reviews)"


def refine_solutions(problem: dict, solutions: dict, reviews: dict, solver_agents: dict) -> dict:
    """Stage 3: returns {solver_id: Refinement}"""
    def make_task(solver_id):
        sol = solutions[solver_id]
        prompt = (
            f"Problem:\n{problem['question']}\n\n"
            f"Your original solution:\nReasoning: {sol.reasoning}\nFinal answer: {sol.final_answer}\n\n"
            f"Peer reviews of your solution:{format_reviews(reviews[solver_id])}\n\n"
            f"Respond to each critique and produce your refined solution."
        )
        return lambda: ask(solver_agents[solver_id], REFINE_SYS, prompt, Refinement, temperature=0.3)

    ids = list(solutions.keys())
    results = run_parallel([make_task(sid) for sid in ids])
    return dict(zip(ids, results))

## Stage 4: final judgment

The Judge receives the full history: original solutions, all peer reviews and the refined solutions.
It has to verify the reasoning itself and pick the solver whose refined answer is most likely correct.
The system's final answer is the winner's refined answer.

In [41]:
JUDGE_SYS = (
    "You are the final Judge in a multi-LLM debate system. You receive three refined solutions to the same "
    "problem, together with the original solutions and the peer reviews. Do not just count votes or trust "
    "confidence scores - verify the reasoning of each refined solution yourself, step by step, and pick the "
    "solver whose refined final answer is most likely correct."
)


def judge_solutions(problem: dict, judge_agent: str, solutions: dict, reviews: dict, refinements: dict) -> JudgeVerdict:
    """Stage 4: the judge picks a winner"""
    prompt = f"Problem:\n{problem['question']}\n"
    for sid in solutions:
        prompt += f"\n================ {sid} ================\n"
        prompt += f"Original reasoning: {solutions[sid].reasoning}\n"
        prompt += f"Original answer: {solutions[sid].final_answer}\n"
        prompt += f"Reviews received:{format_reviews(reviews[sid])}\n"
        prompt += f"Refined reasoning: {refinements[sid].refined_reasoning}\n"
        prompt += f"Refined answer: {refinements[sid].refined_answer}\n"
    prompt += "\nVerify each refined solution and pick the winner."
    return ask(judge_agent, JUDGE_SYS, prompt, JudgeVerdict, temperature=0.0)